In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np
from tqdm import tqdm
import pandas as pd

MODEL_NAME = "microsoft/phi-2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

df = pd.read_csv("cities_true_false.csv")
TRUE_STMTS = df[df['label'] == 1]['statement'].tolist()
FALSE_STMTS = df[df['label'] == 0]['statement'].tolist()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, trust_remote_code=True).to(DEVICE).eval()
num_layers = len(model.model.layers)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [11]:
def get_acts(text, layer):
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    acts = {}
    hook = model.model.layers[layer].register_forward_hook(lambda m,i,o: acts.update({'h': o[0].detach()}))
    with torch.no_grad(): model(**inputs)
    hook.remove()
    return acts['h'][0, -1, :].cpu().numpy()

def get_direction(true_stmts, false_stmts, layer):
    true_acts = np.array([get_acts(s, layer) for s in tqdm(true_stmts, leave=False)])
    false_acts = np.array([get_acts(s, layer) for s in tqdm(false_stmts, leave=False)])
    d = true_acts.mean(0) - false_acts.mean(0)
    return d / np.linalg.norm(d), true_acts, false_acts

def test_acc(true_acts, false_acts, d):
    true_p, false_p = true_acts @ d, false_acts @ d
    thresh = (true_p.mean() + false_p.mean()) / 2
    acc = (np.sum(true_p > thresh) + np.sum(false_p <= thresh)) / (len(true_p) + len(false_p))
    sep = true_p.mean() - false_p.mean()
    return acc, sep, true_p.mean(), false_p.mean()

def generate(text, layer=None, direction=None, scale=0):
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    hook = None
    if direction is not None and scale != 0:
        def hook_fn(m, i, o):
            delta = torch.tensor(direction * scale, device=DEVICE, dtype=o[0].dtype)
            return (o[0] + delta,) + o[1:]
        hook = model.model.layers[layer].register_forward_hook(hook_fn)
    with torch.no_grad(): out = model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    if hook: hook.remove()
    return tokenizer.decode(out[0], skip_special_tokens=True)

def generate_ablated(text, layer, direction):
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    def hook_fn(m, i, o):
        h = o[0]
        d = torch.tensor(direction, device=DEVICE, dtype=h.dtype)
        proj = (h @ d).unsqueeze(-1) * d
        return (h - proj,) + o[1:]
    hook = model.model.layers[layer].register_forward_hook(hook_fn)
    with torch.no_grad(): out = model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    hook.remove()
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
true_train, false_train = TRUE_STMTS[:80], FALSE_STMTS[:80]
true_test, false_test = TRUE_STMTS[80:130], FALSE_STMTS[80:130]

results = []
for layer in range(0, num_layers, 4):
    d, _, _ = get_direction(true_train, false_train, layer)
    _, ta, fa = get_direction(true_test, false_test, layer)
    acc, sep, tm, fm = test_acc(ta, fa, d)
    results.append((layer, acc, sep, d))
    print(f"Layer {layer:2d}: acc={acc:.1%}, sep={sep:.4f}, true_mean={tm:.4f}, false_mean={fm:.4f}")

best = max(results, key=lambda x: x[1])
best_layer, best_acc, best_sep, best_dir = best
print(f"\nBest: layer {best_layer}, acc {best_acc:.1%}, separation {best_sep:.4f}")

test_prompts = [
    "Paris is a city in",
    "Tokyo is located in",
    "London is the capital of",
    "Berlin is in the country of",
]
for scale in [1.0, 5.0, 10.0]:
    print(f"\n--- Scale {scale} ---")
    for p in test_prompts[:2]:
        print(f"'{p}'")
        print(f"  Base: {generate(p)}")
        print(f"  +Dir: {generate(p, best_layer, best_dir, scale)}")
        print(f"  -Dir: {generate(p, best_layer, best_dir, -scale)}")

for p in test_prompts:
    print(f"'{p}'")
    print(f"  Base:    {generate(p)}")
    print(f"  Ablated: {generate_ablated(p, best_layer, best_dir)}")

for s in TRUE_STMTS[:3]:
    proj = get_acts(s, best_layer) @ best_dir
    print(f"TRUE  (proj={proj:.4f}): {s}")
for s in FALSE_STMTS[:3]:
    proj = get_acts(s, best_layer) @ best_dir
    print(f"FALSE (proj={proj:.4f}): {s}")

Model: microsoft/phi-2, Layers: 32, Device: cuda
Data: 729 true, 729 false

=== Layer Search ===


Layer  0: acc=48.0%, sep=-1.4219, true_mean=19.4844, false_mean=20.9062


Layer  4: acc=48.0%, sep=-1.5078, true_mean=6.7500, false_mean=8.2578


Layer  8: acc=48.0%, sep=-0.8750, true_mean=14.3906, false_mean=15.2656


Layer 12: acc=48.0%, sep=2.2969, true_mean=16.2969, false_mean=14.0000


Layer 16: acc=48.0%, sep=6.1016, true_mean=20.6094, false_mean=14.5078


Layer 20: acc=48.0%, sep=4.9844, true_mean=31.8438, false_mean=26.8594


Layer 24: acc=48.0%, sep=4.9062, true_mean=51.7500, false_mean=46.8438


Layer 28: acc=48.0%, sep=4.9062, true_mean=53.4688, false_mean=48.5625

Best: layer 0, acc 48.0%, separation -1.4219

=== Activation Addition ===

--- Scale 1.0 ---
'Paris is a city in'
  Base: Paris is a city in France.

  +Dir: Paris is a city in France.

  -Dir: Paris is a city in France.

'Tokyo is located in'
  Base: Tokyo is located in Japan.

  +Dir: Tokyo is located in Japan.

  -Dir: Tokyo is located in Japan.


--- Scale 5.0 ---
'Paris is a city in'
  Base: Paris is a city in France.

  +Dir: Paris is a city in France.

  -Dir: Paris is a city in France.

'Tokyo is located in'
  Base: Tokyo is located in Japan.

  +Dir: Tokyo is located in Japan.

  -Dir: Tokyo is located in Japan.


--- Scale 10.0 ---
'Paris is a city in'
  Base: Paris is a city in France.

  +Dir: Paris is a city in France.

  -Dir: Paris is a city in France.

'Tokyo is located in'
  Base: Tokyo is located in Japan.

  +Dir: Tokyo is located in Japan.

  -Dir: Tokyo is located in Japan.


=== Directional Ab